# **Problem Statement**

## Business Context

The EdTech industry has been surging in the past decade immensely, and according to a forecast, the Online Education market would be worth $286.62 bn by 2023 with a compound annual growth rate (CAGR) of 10.26% from 2018 to 2023. The modern era of online education has enforced a lot in its growth and expansion beyond any limit. Due to having many dominant features like ease of information sharing, personalized learning experience, transparency of assessment, etc, it is now preferable to traditional education.

In the present scenario due to the Covid-19, the online education sector has witnessed rapid growth and is attracting a lot of new customers. Due to this rapid growth, many new companies have emerged in this industry. With the availability and ease of use of digital marketing resources, companies can reach out to a wider audience with their offerings. The customers who show interest in these offerings are termed as leads. There are various sources of obtaining leads for Edtech companies, like

* The customer interacts with the marketing front on social media or other online platforms.
* The customer browses the website/app and downloads the brochure
* The customer connects through emails for more information.

The company then nurtures these leads and tries to convert them to paid customers. For this, the representative from the organization connects with the lead on call or through email to share further details.

## Objective

ExtraaLearn is an initial stage startup that offers programs on cutting-edge technologies to students and professionals to help them upskill/reskill. With a large number of leads being generated on a regular basis, one of the issues faced by ExtraaLearn is to identify which of the leads are more likely to convert so that they can allocate resources accordingly. You, as a data scientist at ExtraaLearn, have been provided the leads data to:
* Analyze and build an ML model to help identify which leads are more likely to convert to paid customers,
* Find the factors driving the lead conversion process
* Create a profile of the leads which are likely to convert

## Data Description

The data contains the different attributes of leads and their interaction details with ExtraaLearn. The detailed data dictionary is given below.

* **ID**: ID of the lead.
* **age**: Age of the lead.
* **current_occupation**: Current occupation of the lead. Values include 'Professional','Unemployed',and 'Student'.
* **first_interaction**: How did the lead first interacted with ExtraaLearn. Values include 'Website', 'Mobile App'.
* **profile_completed**: What percentage of profile has been filled by the lead on the website/mobile app. Values include Low - (0-50%), Medium - (50-75%), High (75-100%).
* **website_visits**: How many times has a lead visited the website.
* **time_spent_on_website**: Total time spent on the website in seconds.
* **page_views_per_visit**: Average number of pages on the website viewed during the visits.
* **last_activity**: Last interaction between the lead and ExtraaLearn.
    * Email Activity: Seeking for details about program through email, Representative shared information with lead like brochure of program , etc.
    * Phone Activity: Had a Phone Conversation with representative, Had conversation over SMS with representative, etc.
    * Website Activity: Interacted on live chat with representative, Updated profile on website, etc.

* **print_media_type1**: Flag indicating whether the lead had seen the ad of ExtraaLearn in the Newspaper.
* **print_media_type2**: Flag indicating whether the lead had seen the ad of ExtraaLearn in the Magazine.
* **digital_media**: Flag indicating whether the lead had seen the ad of ExtraaLearn on the digital platforms.
* **educational_channels**: Flag indicating whether the lead had heard about ExtraaLearn in the education channels like online forums, discussion threads, educational websites, etc.
* **referral**: Flag indicating whether the lead had heard about ExtraaLearn through reference.
* **status**: Flag indicating whether the lead was converted to a paid customer or not.


# **Installing and Importing the Necessary libraries**

In [ ]:
#Installing the libraries with the specified versions
!pip install numpy==2.0.2 pandas==2.2.2 scikit-learn==1.6.1 matplotlib==3.10.0 seaborn==0.13.2 joblib==1.4.2 xgboost==2.1.4 requests==2.32.3 huggingface_hub==0.30.1 -q

**Note:**

- After running the above cell, kindly restart the notebook kernel (for Jupyter Notebook) or runtime (for Google Colab) and run all cells sequentially from the next cell.

- On executing the above line of code, you might see a warning regarding package dependencies. This error message can be ignored as the above code ensures that all necessary libraries and their dependencies are maintained to successfully execute the code in this notebook.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

# Libraries to help with reading and manipulating data
import numpy as np
import pandas as pd

# For splitting the dataset
from sklearn.model_selection import train_test_split

# Libaries to help with data visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Removes the limit for the number of displayed columns
pd.set_option("display.max_columns", None)
# Sets the limit for the number of displayed rows
pd.set_option("display.max_rows", 100)


# Libraries different ensemble classifiers
from sklearn.ensemble import (
    BaggingClassifier,
    RandomForestClassifier,
    AdaBoostClassifier,
    GradientBoostingClassifier,
)
from xgboost import XGBClassifier
from sklearn.tree import DecisionTreeClassifier

# Libraries to get different metric scores
from sklearn.metrics import (
    confusion_matrix,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
)

# To create the pipeline
from sklearn.compose import ColumnTransformer, make_column_transformer
from sklearn.pipeline import make_pipeline,Pipeline

# To tune different models and standardize
from sklearn.model_selection import GridSearchCV
from sklearn.preprocessing import StandardScaler,OneHotEncoder

# To serialize the model
import joblib

# os related functionalities
import os

# API request
import requests

# for hugging face space authentication to upload files
from huggingface_hub import login, HfApi

# **Loading the dataset**

In [ ]:
# uncomment and run the below code snippets if the dataset is present in the Google Drive
# from google.colab import drive
# drive.mount('/content/drive')

## **Overview of the dataset**

In [ ]:
# Loading the dataset
# If running on Google Colab, upload ExtraaLearn.csv to the session files before running this cell.
data = pd.read_csv("ExtraaLearn.csv")

print(f"There are {data.shape[0]} rows and {data.shape[1]} columns in the dataset.")
display(data.head())

display(data.info())
display(data.describe(include="all").T)

print("Missing values in each column:")
display(data.isnull().sum())

print("Number of duplicate rows:", data.duplicated().sum())

print("Target variable distribution:")
display(pd.DataFrame({
    "Count": data["status"].value_counts().sort_index(),
    "Percentage": (data["status"].value_counts(normalize=True).sort_index() * 100).round(2),
}))


# **Exploratory Data Analysis**

**The below functions need to be defined to carry out the EDA.**

In [ ]:
def histogram_boxplot(data, feature, figsize=(15, 10), kde=False, bins=None):
    """
    Boxplot and histogram combined

    data: dataframe
    feature: dataframe column
    figsize: size of figure (default (15,10))
    kde: whether to show the density curve (default False)
    bins: number of bins for histogram (default None)
    """
    f2, (ax_box2, ax_hist2) = plt.subplots(
        nrows=2,  # Number of rows of the subplot grid= 2
        sharex=True,  # x-axis will be shared among all subplots
        gridspec_kw={"height_ratios": (0.25, 0.75)},
        figsize=figsize,
    )  # creating the 2 subplots
    sns.boxplot(
        data=data, x=feature, ax=ax_box2, showmeans=True, color="violet"
    )  # boxplot will be created and a triangle will indicate the mean value of the column
    sns.histplot(
        data=data, x=feature, kde=kde, ax=ax_hist2, bins=bins
    ) if bins else sns.histplot(
        data=data, x=feature, kde=kde, ax=ax_hist2
    )  # For histogram
    ax_hist2.axvline(
        data[feature].mean(), color="green", linestyle="--"
    )  # Add mean to the histogram
    ax_hist2.axvline(
        data[feature].median(), color="black", linestyle="-"
    )  # Add median to the histogram

In [ ]:
# function to create labeled barplots


def labeled_barplot(data, feature, perc=False, n=None):
    """
    Barplot with percentage at the top

    data: dataframe
    feature: dataframe column
    perc: whether to display percentages instead of count (default is False)
    n: displays the top n category levels (default is None, i.e., display all levels)
    """

    total = len(data[feature])  # length of the column
    count = data[feature].nunique()
    if n is None:
        plt.figure(figsize=(count + 2, 6))
    else:
        plt.figure(figsize=(n + 2, 6))

    plt.xticks(rotation=90, fontsize=15)
    ax = sns.countplot(
        data=data,
        x=feature,
        order=data[feature].value_counts().index[:n],
    )

    for p in ax.patches:
        if perc == True:
            label = "{:.1f}%".format(
                100 * p.get_height() / total
            )  # percentage of each class of the category
        else:
            label = p.get_height()  # count of each level of the category

        x = p.get_x() + p.get_width() / 2  # width of the plot
        y = p.get_height()  # height of the plot

        ax.annotate(
            label,
            (x, y),
            ha="center",
            va="center",
            size=12,
            xytext=(0, 5),
            textcoords="offset points",
        )  # annotate the percentage

    plt.show()  # show the plot

In [ ]:
def stacked_barplot(data, predictor, target):
    """
    Print the category counts and plot a stacked bar chart

    data: dataframe
    predictor: independent variable
    target: target variable
    """
    count = data[predictor].nunique()
    sorter = data[target].value_counts().index[-1]
    tab1 = pd.crosstab(data[predictor], data[target], margins=True).sort_values(
        by=sorter, ascending=False
    )
    print(tab1)
    print("-" * 120)
    tab = pd.crosstab(data[predictor], data[target], normalize="index").sort_values(
        by=sorter, ascending=False
    )
    tab.plot(kind="bar", stacked=True, figsize=(count + 5, 5))
    plt.legend(
        loc="lower left", frameon=False,
    )
    plt.legend(loc="upper left", bbox_to_anchor=(1, 1))
    plt.show()

In [ ]:
### function to plot distributions wrt target


def distribution_plot_wrt_target(data, predictor, target):

    fig, axs = plt.subplots(2, 2, figsize=(12, 10))

    target_uniq = data[target].unique()

    axs[0, 0].set_title("Distribution of target for target=" + str(target_uniq[0]))
    sns.histplot(
        data=data[data[target] == target_uniq[0]],
        x=predictor,
        kde=True,
        ax=axs[0, 0],
        color="teal",
        stat="density",
    )

    axs[0, 1].set_title("Distribution of target for target=" + str(target_uniq[1]))
    sns.histplot(
        data=data[data[target] == target_uniq[1]],
        x=predictor,
        kde=True,
        ax=axs[0, 1],
        color="orange",
        stat="density",
    )

    axs[1, 0].set_title("Boxplot w.r.t target")
    sns.boxplot(data=data, x=target, y=predictor, ax=axs[1, 0])

    axs[1, 1].set_title("Boxplot (without outliers) w.r.t target")
    sns.boxplot(
        data=data,
        x=target,
        y=predictor,
        ax=axs[1, 1],
        showfliers=False,
    )

    plt.tight_layout()
    plt.show()

### Univariate Analysis

In [ ]:
numerical_columns = ["age", "website_visits", "time_spent_on_website", "page_views_per_visit"]
categorical_columns = [
    "current_occupation",
    "first_interaction",
    "profile_completed",
    "last_activity",
    "print_media_type1",
    "print_media_type2",
    "digital_media",
    "educational_channels",
    "referral",
]

for column in numerical_columns:
    histogram_boxplot(data, column, kde=True)

for column in categorical_columns + ["status"]:
    labeled_barplot(data, column, perc=True)


### Bivariate Analysis

In [ ]:
plt.figure(figsize=(8, 5))
sns.heatmap(data[numerical_columns + ["status"]].corr(), annot=True, cmap="YlGnBu", fmt=".2f")
plt.title("Correlation Heatmap")
plt.show()

for column in numerical_columns:
    distribution_plot_wrt_target(data, column, "status")

for column in categorical_columns:
    stacked_barplot(data, column, "status")

conversion_by_category = []
for column in categorical_columns:
    temp = (
        data.groupby(column)["status"]
        .agg(Count="count", Conversion_Rate="mean")
        .reset_index()
    )
    temp["Feature"] = column
    temp["Conversion_Rate"] = (temp["Conversion_Rate"] * 100).round(2)
    conversion_by_category.append(temp[["Feature", column, "Count", "Conversion_Rate"]])

for table in conversion_by_category:
    display(table.sort_values("Conversion_Rate", ascending=False))


# **Data Preprocessing**

## Outlier Check

In [ ]:
outlier_summary = []
for column in numerical_columns:
    q1 = data[column].quantile(0.25)
    q3 = data[column].quantile(0.75)
    iqr = q3 - q1
    lower_bound = q1 - 1.5 * iqr
    upper_bound = q3 + 1.5 * iqr
    outlier_count = data[(data[column] < lower_bound) | (data[column] > upper_bound)].shape[0]
    outlier_summary.append(
        {
            "Feature": column,
            "Lower Bound": lower_bound,
            "Upper Bound": upper_bound,
            "Outlier Count": outlier_count,
            "Outlier %": round(outlier_count / data.shape[0] * 100, 2),
        }
    )

outlier_summary = pd.DataFrame(outlier_summary)
display(outlier_summary)

for column in numerical_columns:
    plt.figure(figsize=(8, 3))
    sns.boxplot(data=data, x=column)
    plt.title(f"Outlier Check - {column}")
    plt.show()

# The selected tree-based models are robust to outliers, so no outlier capping is applied.


## Data Preparation for modeling

- We want to predict which lead is more likely to be converted.
- Before we proceed to build a model, we'll have to encode categorical features.
- We'll split the data into train and test to be able to evaluate the model that we build on the train data.

In [ ]:
X = data.drop(["ID", "status"], axis=1)
y = data["status"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.30,
    random_state=1,
    stratify=y,
)

print("Training data shape:", X_train.shape)
print("Test data shape:", X_test.shape)
print("Training target distribution:")
display(y_train.value_counts(normalize=True).sort_index().mul(100).round(2))
print("Test target distribution:")
display(y_test.value_counts(normalize=True).sort_index().mul(100).round(2))


## **Data Pre-processing Pipeline**

In [ ]:
numeric_features = X.select_dtypes(include=np.number).columns.tolist()
categorical_features = X.select_dtypes(include=["object", "category"]).columns.tolist()

preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features),
    ],
    remainder="drop",
)

print("Numeric features:", numeric_features)
print("Categorical features:", categorical_features)


# **Model Building**

## Model Evaluation Criterion

The business objective is to identify leads that are likely to become paid customers so that ExtraaLearn can prioritize follow-up efforts. Missing a good lead may mean lost revenue, while marking too many weak leads as high priority can waste the sales team's time.

For this reason, **F1-score** is used as the primary model selection metric because it balances **precision** and **recall** for the converted class. Recall is also monitored closely because the company should avoid missing leads with a strong chance of conversion.


## Define functions for Model Evaluation

In [ ]:
# Function to compute different metrics to check performance of a classification model
def model_performance_classification(model, predictors, target):
    """
    Function to compute different metrics to check classifier model performance

    model: classifier
    predictors: independent variables
    target: dependent variable
    """

    pred = model.predict(predictors)

    acc = accuracy_score(target, pred)
    recall = recall_score(target, pred)
    precision = precision_score(target, pred)
    f1 = f1_score(target, pred)

    df_perf = pd.DataFrame(
        {"Accuracy": acc, "Recall": recall, "Precision": precision, "F1": f1},
        index=[0],
    )

    return df_perf


def confusion_matrix_sklearn(model, predictors, target):
    """Plot a confusion matrix for a classification model."""
    pred = model.predict(predictors)
    cm = confusion_matrix(target, pred)
    plt.figure(figsize=(5, 4))
    sns.heatmap(
        cm,
        annot=True,
        fmt="d",
        cmap="Blues",
        xticklabels=["Not Converted", "Converted"],
        yticklabels=["Not Converted", "Converted"],
    )
    plt.ylabel("Actual")
    plt.xlabel("Predicted")
    plt.show()


## Random Forest Model

In [ ]:
rf_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "model",
            RandomForestClassifier(
                random_state=1,
                n_estimators=200,
                class_weight="balanced",
            ),
        ),
    ]
)

rf_model.fit(X_train, y_train)


### Checking model performance on training set

In [ ]:
rf_train_perf = model_performance_classification(rf_model, X_train, y_train)
display(rf_train_perf)
confusion_matrix_sklearn(rf_model, X_train, y_train)


### Checking model performance on test set

In [ ]:
rf_test_perf = model_performance_classification(rf_model, X_test, y_test)
display(rf_test_perf)
confusion_matrix_sklearn(rf_model, X_test, y_test)


## AdaBoost Classifier

In [ ]:
adaboost_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "model",
            AdaBoostClassifier(
                estimator=DecisionTreeClassifier(max_depth=1, random_state=1),
                random_state=1,
                n_estimators=100,
                learning_rate=0.1,
            ),
        ),
    ]
)

adaboost_model.fit(X_train, y_train)


### Checking model performance on training set

In [ ]:
adaboost_train_perf = model_performance_classification(adaboost_model, X_train, y_train)
display(adaboost_train_perf)
confusion_matrix_sklearn(adaboost_model, X_train, y_train)


### Checking model performance on test set

In [ ]:
adaboost_test_perf = model_performance_classification(adaboost_model, X_test, y_test)
display(adaboost_test_perf)
confusion_matrix_sklearn(adaboost_model, X_test, y_test)


# **Model Performance Improvement - Hyperparameter Tuning**

## Hyperparameter Tuning - Random Forest

In [ ]:
rf_tuned_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", RandomForestClassifier(random_state=1, class_weight="balanced")),
    ]
)

rf_parameters = {
    "model__n_estimators": [100, 200],
    "model__max_depth": [4, 6, 8, None],
    "model__min_samples_split": [2, 5, 10],
    "model__min_samples_leaf": [1, 2, 4],
    "model__max_features": ["sqrt", None],
}

rf_grid = GridSearchCV(
    estimator=rf_tuned_model,
    param_grid=rf_parameters,
    scoring="f1",
    cv=5,
    n_jobs=-1,
)

rf_grid.fit(X_train, y_train)

print("Best parameters:", rf_grid.best_params_)
print("Best cross-validation F1-score:", rf_grid.best_score_)

tuned_rf_model = rf_grid.best_estimator_


### Checking model performance on training set

In [ ]:
tuned_rf_train_perf = model_performance_classification(tuned_rf_model, X_train, y_train)
display(tuned_rf_train_perf)
confusion_matrix_sklearn(tuned_rf_model, X_train, y_train)


### Checking model performance on test set

In [ ]:
tuned_rf_test_perf = model_performance_classification(tuned_rf_model, X_test, y_test)
display(tuned_rf_test_perf)
confusion_matrix_sklearn(tuned_rf_model, X_test, y_test)


## Hyperparameter Tuning - AdaBoost Classifier

In [ ]:
adaboost_tuned_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", AdaBoostClassifier(random_state=1)),
    ]
)

adaboost_parameters = {
    "model__estimator": [
        DecisionTreeClassifier(max_depth=1, random_state=1),
        DecisionTreeClassifier(max_depth=2, random_state=1),
        DecisionTreeClassifier(max_depth=3, random_state=1),
    ],
    "model__n_estimators": [50, 100, 150],
    "model__learning_rate": [0.01, 0.05, 0.1, 0.5, 1.0],
}

adaboost_grid = GridSearchCV(
    estimator=adaboost_tuned_model,
    param_grid=adaboost_parameters,
    scoring="f1",
    cv=5,
    n_jobs=-1,
)

adaboost_grid.fit(X_train, y_train)

print("Best parameters:", adaboost_grid.best_params_)
print("Best cross-validation F1-score:", adaboost_grid.best_score_)

tuned_adaboost_model = adaboost_grid.best_estimator_


### Checking model performance on training set

In [ ]:
tuned_adaboost_train_perf = model_performance_classification(tuned_adaboost_model, X_train, y_train)
display(tuned_adaboost_train_perf)
confusion_matrix_sklearn(tuned_adaboost_model, X_train, y_train)


### Checking model performance on test set

In [ ]:
tuned_adaboost_test_perf = model_performance_classification(tuned_adaboost_model, X_test, y_test)
display(tuned_adaboost_test_perf)
confusion_matrix_sklearn(tuned_adaboost_model, X_test, y_test)


# **Model Performance Comparison, Final Model Selection, and Serialization**

In [ ]:
models_train_comp_df = pd.concat(
    [
        rf_train_perf.T,
        adaboost_train_perf.T,
        tuned_rf_train_perf.T,
        tuned_adaboost_train_perf.T,
    ],
    axis=1,
)
models_train_comp_df.columns = [
    "Random Forest",
    "AdaBoost",
    "Tuned Random Forest",
    "Tuned AdaBoost",
]
print("Training performance comparison")
display(models_train_comp_df)

models_test_comp_df = pd.concat(
    [
        rf_test_perf.T,
        adaboost_test_perf.T,
        tuned_rf_test_perf.T,
        tuned_adaboost_test_perf.T,
    ],
    axis=1,
)
models_test_comp_df.columns = [
    "Random Forest",
    "AdaBoost",
    "Tuned Random Forest",
    "Tuned AdaBoost",
]
print("Test performance comparison")
display(models_test_comp_df)

if tuned_adaboost_test_perf.loc[0, "F1"] >= tuned_rf_test_perf.loc[0, "F1"]:
    final_model = tuned_adaboost_model
    final_model_name = "Tuned AdaBoost"
else:
    final_model = tuned_rf_model
    final_model_name = "Tuned Random Forest"

print("Final selected model:", final_model_name)

try:
    feature_names = final_model.named_steps["preprocessor"].get_feature_names_out()
    importances = final_model.named_steps["model"].feature_importances_
    importance_df = (
        pd.DataFrame({"Feature": feature_names, "Importance": importances})
        .sort_values("Importance", ascending=False)
        .head(20)
    )
    display(importance_df)
except Exception as exc:
    print("Feature importance could not be generated:", exc)

os.makedirs("backend_files", exist_ok=True)

model_bundle = {
    "model": final_model,
    "features": X.columns.tolist(),
    "categorical_features": categorical_features,
    "numeric_features": numeric_features,
    "target": "status",
    "model_name": final_model_name,
}

joblib.dump(model_bundle, "model.joblib")
joblib.dump(model_bundle, "backend_files/model.joblib")
print("Model serialized as model.joblib and backend_files/model.joblib")


# **Deployment - Backend**

## Flask Web Framework


In [ ]:
# Create deployment folders before writing app files
os.makedirs("backend_files", exist_ok=True)
os.makedirs("frontend_files", exist_ok=True)


In [ ]:
%%writefile backend_files/app.py
from pathlib import Path

import joblib
import pandas as pd
from flask import Flask, jsonify, request


MODEL_PATH = Path(__file__).with_name("model.joblib")
MODEL_BUNDLE = joblib.load(MODEL_PATH)
MODEL = MODEL_BUNDLE["model"]
FEATURES = MODEL_BUNDLE["features"]

app = Flask(__name__)


def _payload_to_frame(payload):
    if isinstance(payload, dict):
        records = payload.get("data", payload)
    elif isinstance(payload, list):
        records = payload
    else:
        raise ValueError("Payload must be a JSON object or a list of objects.")

    if isinstance(records, dict):
        records = [records]

    frame = pd.DataFrame(records)
    missing = [column for column in FEATURES if column not in frame.columns]
    if missing:
        raise ValueError(f"Missing required field(s): {', '.join(missing)}")

    return frame[FEATURES]


@app.get("/")
def health_check():
    return jsonify(
        {
            "status": "ok",
            "model": MODEL_BUNDLE.get("model_name", "ExtraaLearn lead conversion model"),
            "features": FEATURES,
        }
    )


@app.post("/predict")
def predict():
    try:
        payload = request.get_json(force=True)
        input_data = _payload_to_frame(payload)
        predictions = MODEL.predict(input_data).astype(int).tolist()

        if hasattr(MODEL, "predict_proba"):
            probabilities = MODEL.predict_proba(input_data)[:, 1].round(4).tolist()
        else:
            probabilities = [None] * len(predictions)

        results = [
            {
                "prediction": prediction,
                "prediction_label": "Converted" if prediction == 1 else "Not Converted",
                "conversion_probability": probability,
            }
            for prediction, probability in zip(predictions, probabilities)
        ]
        return jsonify({"results": results})
    except Exception as exc:
        return jsonify({"error": str(exc)}), 400


if __name__ == "__main__":
    app.run(host="0.0.0.0", port=7860)


## Dependencies File

In [ ]:
%%writefile backend_files/requirements.txt
flask==3.0.3
joblib==1.4.2
numpy==2.0.2
pandas==2.2.2
scikit-learn==1.6.1


## Dockerfile

In [ ]:
%%writefile backend_files/Dockerfile
FROM python:3.9-slim

WORKDIR /app

COPY . .

RUN pip install --no-cache-dir -r requirements.txt

EXPOSE 7860

CMD ["python", "app.py"]


## Setting up a Hugging Face Docker Space for the Backend

**Note**: We are creating a Hugging Face Docker Space for our backend using the Hugging Face Hub API. This automates the space creation process and enables seamless deployment of our Flask app.

In [ ]:
# Authenticate with Hugging Face.
# In Colab, add HF_TOKEN as a secret. You can also set it as an environment variable.
# If HF_TOKEN is not found, the cell will ask you to paste the token securely.
import getpass

hf_token = os.getenv("HF_TOKEN")

try:
    from google.colab import userdata
    hf_token = hf_token or userdata.get("HF_TOKEN")
except Exception:
    pass

if not hf_token:
    hf_token = getpass.getpass("Enter your Hugging Face token: ")

if not hf_token:
    raise ValueError("Please provide HF_TOKEN before creating or uploading to Hugging Face Spaces.")

login(token=hf_token)
api = HfApi()


**Note :** If you were trying with different names, be cautious when using a underscore `_` in space names, such as `frontend_space`, as it can cause exceptions when accessing the API. Always use an hyphen `-` instead, like `frontend-space`.

In [ ]:
# Replace this with your Hugging Face username.
HF_USERNAME = "your-huggingface-username"
BACKEND_SPACE_NAME = "extraalearn-backend"
backend_space_id = f"{HF_USERNAME}/{BACKEND_SPACE_NAME}"

api.create_repo(
    repo_id=backend_space_id,
    repo_type="space",
    space_sdk="docker",
    exist_ok=True,
    private=False,
)

print("Backend Space:", backend_space_id)
print("Backend API URL:", f"https://{HF_USERNAME}-{BACKEND_SPACE_NAME}.hf.space")


## Uploading Files to Hugging Face Space (Docker Space)

**Note**: Before running the code below, ensure that the serialized ML model has been uploaded in to `backend_files` folder.

In [ ]:
# Ensure backend_files contains app.py, requirements.txt, Dockerfile, and model.joblib before running this cell.
import os
import shutil

os.makedirs("backend_files", exist_ok=True)

if not os.path.exists("backend_files/model.joblib") and os.path.exists("model.joblib"):
    shutil.copy("model.joblib", "backend_files/model.joblib")

required_backend_files = ["app.py", "requirements.txt", "Dockerfile", "model.joblib"]
missing_backend_files = [
    file_name for file_name in required_backend_files
    if not os.path.exists(os.path.join("backend_files", file_name))
]

if missing_backend_files:
    raise FileNotFoundError(
        "Missing backend file(s): " + ", ".join(missing_backend_files) +
        ". Rerun the app/Docker/requirements/model serialization cells before uploading."
    )

print("Backend files ready:", os.listdir("backend_files"))
print("Model size:", os.path.getsize("backend_files/model.joblib"), "bytes")

api.upload_folder(
    folder_path="backend_files",
    repo_id=backend_space_id,
    repo_type="space",
)

print("Backend files uploaded successfully.")


# **Deployment - Frontend**

## Points to note before executing the below cells
- Create a Streamlit space on Hugging Face by following the instructions provided on the content page titled **`Creating Spaces and Adding Secrets in Hugging Face`** from Week 1

## Streamlit for Interactive UI

In [ ]:
%%writefile frontend_files/app.py
import os

import pandas as pd
import requests
import streamlit as st


DEFAULT_BACKEND_URL = os.getenv("BACKEND_URL", "").rstrip("/")

st.set_page_config(page_title="ExtraaLearn Lead Conversion", page_icon="EL", layout="centered")
st.title("ExtraaLearn Lead Conversion")

backend_url = st.sidebar.text_input("Backend URL", value=DEFAULT_BACKEND_URL)

tab_single, tab_batch = st.tabs(["Single Lead", "Batch Scoring"])

with tab_single:
    age = st.number_input("Age", min_value=18, max_value=80, value=35, step=1)
    current_occupation = st.selectbox("Current Occupation", ["Professional", "Unemployed", "Student"])
    first_interaction = st.selectbox("First Interaction", ["Website", "Mobile App"])
    profile_completed = st.selectbox("Profile Completed", ["Low", "Medium", "High"], index=2)
    website_visits = st.number_input("Website Visits", min_value=0, max_value=50, value=5, step=1)
    time_spent_on_website = st.number_input(
        "Time Spent on Website", min_value=0, max_value=5000, value=700, step=10
    )
    page_views_per_visit = st.number_input(
        "Page Views per Visit", min_value=0.0, max_value=25.0, value=3.0, step=0.1
    )
    last_activity = st.selectbox(
        "Last Activity", ["Website Activity", "Email Activity", "Phone Activity"]
    )
    print_media_type1 = st.selectbox("Newspaper Ad", ["No", "Yes"])
    print_media_type2 = st.selectbox("Magazine Ad", ["No", "Yes"])
    digital_media = st.selectbox("Digital Media Ad", ["No", "Yes"])
    educational_channels = st.selectbox("Educational Channels", ["No", "Yes"])
    referral = st.selectbox("Referral", ["No", "Yes"])

    payload = {
        "age": age,
        "current_occupation": current_occupation,
        "first_interaction": first_interaction,
        "profile_completed": profile_completed,
        "website_visits": website_visits,
        "time_spent_on_website": time_spent_on_website,
        "page_views_per_visit": page_views_per_visit,
        "last_activity": last_activity,
        "print_media_type1": print_media_type1,
        "print_media_type2": print_media_type2,
        "digital_media": digital_media,
        "educational_channels": educational_channels,
        "referral": referral,
    }

    if st.button("Predict Conversion", type="primary"):
        if not backend_url:
            st.error("Backend URL is required.")
        else:
            response = requests.post(f"{backend_url}/predict", json=payload, timeout=30)
            if response.ok:
                result = response.json()["results"][0]
                probability = result.get("conversion_probability")
                st.metric("Prediction", result["prediction_label"])
                if probability is not None:
                    st.metric("Conversion Probability", f"{probability:.1%}")
            else:
                st.error(response.json().get("error", response.text))

with tab_batch:
    uploaded_file = st.file_uploader("Upload Leads CSV", type=["csv"])
    if uploaded_file is not None:
        batch_data = pd.read_csv(uploaded_file)
        st.dataframe(batch_data.head(), use_container_width=True)

        if st.button("Score Batch"):
            if not backend_url:
                st.error("Backend URL is required.")
            else:
                records = batch_data.to_dict(orient="records")
                response = requests.post(f"{backend_url}/predict", json={"data": records}, timeout=60)
                if response.ok:
                    results = pd.DataFrame(response.json()["results"])
                    scored_data = pd.concat([batch_data.reset_index(drop=True), results], axis=1)
                    st.dataframe(scored_data, use_container_width=True)
                    st.download_button(
                        "Download Scores",
                        scored_data.to_csv(index=False).encode("utf-8"),
                        "extraalearn_predictions.csv",
                        "text/csv",
                    )
                else:
                    st.error(response.json().get("error", response.text))


## Dependencies File

In [ ]:
%%writefile frontend_files/requirements.txt
streamlit==1.44.1
pandas==2.2.2
requests==2.32.3


## Dockerfile

In [ ]:
%%writefile frontend_files/Dockerfile
FROM python:3.9-slim

WORKDIR /app

COPY . .

RUN pip3 install --no-cache-dir -r requirements.txt

EXPOSE 8501

CMD ["streamlit", "run", "app.py", "--server.port=8501", "--server.address=0.0.0.0", "--server.enableXsrfProtection=false"]


## Uploading Files to Hugging Face Space (Streamlit Space)

In [ ]:
# Create a Streamlit Space manually on Hugging Face or use the code below.
# Add BACKEND_URL as a Space secret, for example:
# https://your-huggingface-username-extraalearn-backend.hf.space

FRONTEND_SPACE_NAME = "extraalearn-frontend"
frontend_space_id = f"{HF_USERNAME}/{FRONTEND_SPACE_NAME}"

api.create_repo(
    repo_id=frontend_space_id,
    repo_type="space",
    space_sdk="docker",
    exist_ok=True,
    private=False,
)

api.upload_folder(
    folder_path="frontend_files",
    repo_id=frontend_space_id,
    repo_type="space",
)

print("Frontend files uploaded successfully.")
print("Frontend Space:", frontend_space_id)


# **Actionable Insights and Business Recommendations**

- The dataset contains **4,612 leads** with no missing values and no duplicate rows. About **29.86%** of leads converted, so the target has moderate class imbalance.
- Leads whose first interaction was through the **Website** converted at about **45.59%**, while Mobile App leads converted at about **10.53%**. This suggests the website journey is currently much stronger for acquisition and conversion.
- Leads with **High profile completion** converted at about **41.78%**, compared with **18.88%** for Medium completion and **7.48%** for Low completion. Profile completion is one of the strongest practical indicators of intent.
- **Referral leads** converted at about **67.74%**, much higher than non-referral leads. ExtraaLearn should build referral campaigns and prioritize referred leads quickly.
- Converted leads spent substantially more time on the website, with an average of about **1,068 seconds** compared with **577 seconds** for non-converted leads. Time spent on the website is also one of the most important model drivers.
- The final selected model is the tuned ensemble model with the best test F1-score. It can be used by the sales team to rank leads by conversion probability and focus first on leads with higher predicted probability.
- Recommended next steps are to prioritize high-profile-completion website leads, create faster outreach for referral leads, improve the Mobile App onboarding journey, and use batch scoring regularly to refresh lead priority lists.


#<font size=6 color='blue'>Power Ahead</font>
___